# 第7回 AR(1) モデルによる予想


この回では、東京17時のドル・円レート（1ドルが何円か）について、前の値から次の値を予想する式を作る。

まず、第5回で調べたことを振り返っておこう。第5回では、レートが前の観測日から何円動いたか（前日からの変化）を毎日求め、ある日の変化とその次の日の変化の関係を調べた。

- 向き：ある日に円安へ動いても、次の日も円安になるとは限らない。ある日の変化と次の日の変化の相関係数は −0.019 とほぼ0で、その日の動きの向きから次の日の向きは読めなかった。
- 大きさ：円安か円高かを問わず何円動いたか（変化の大きさ）で見ると、ある日と次の日の相関係数は0.166だった。大きく動いた日の次の日は、また大きく動きやすい。
- 予想の誤差：今日のレートをそのまま次の日の予想にすると、2024年の誤差の大きさの平均（当日のレートと予想の差を、符号を外して平均した値）は1ドルあたり0.70円だった。

この回では、前の値で次の値を予想する式（AR(1) モデル）を2つの系列に当てはめる。1つは第6回で求めた月ごとの変動の大きさ（その月の日々の動きの標準偏差）、もう1つは日々のレートの前日からの変化である。それぞれの系列で、前の値や過去の平均をそのまま使う予想より誤差が小さくなるかを比べよう。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下の2つのセルを実行する。2つ目のセルは、直線を求める計算に使う `statsmodels` を読み込む。


In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


In [ ]:
import sys
if sys.platform == "emscripten":
    import pyodide_js
    await pyodide_js.loadPackage("statsmodels")
import statsmodels.api as sm


## 第6回 演習1の解説

第6回の演習1から、問3と追加問題を振り返ろう。まず第6回と同じ手順で、東京17時のレート `rate` と対数変化率 `log_pct` を作り直す。


In [ ]:
boj = load_boj_fx()
rate = boj["usd_jpy"].dropna()
log_pct = 100 * np.log(rate).diff()
print(f"観測日：{len(rate)} 日")


### 問3：ドル・円とユーロ・ドルの比較

問3は、FRBのニューヨーク正午のドル・円とユーロ・ドルのどちらが大きく動いたといえるかを、前日からの変化の標準偏差と、対数変化率の標準偏差の2通りで比べる問いだった。2つの通貨の組について、26年分の2つの標準偏差を並べよう。


In [ ]:
fed = load_fed_fx()
usd_jpy = fed["usd_jpy"].dropna()
eur_usd = fed["eur_usd"].dropna()

pd.DataFrame({
    "変化の標準偏差（それぞれの単位）": [usd_jpy.diff().std(), eur_usd.diff().std()],
    "対数変化率の標準偏差（%）": [(100 * np.log(usd_jpy).diff()).std(), (100 * np.log(eur_usd).diff()).std()],
}, index=["ドル・円（円／ドル）", "ユーロ・ドル（ドル／ユーロ）"])


変化の標準偏差はドル・円が0.708円、ユーロ・ドルが0.007ドルで、ドル・円のほうがずっと大きく見える。しかし2つの値は円とドルという別の単位で測っていて、ユーロ・ドルは1ユーロが1ドル前後なので、1日の変化も小さな数になる。単位のそろった対数変化率の標準偏差は0.623%と0.579%で、割合で見ると2つの通貨の組は同じくらいの大きさで動いていた。

年ごとに比べると、どちらが大きいかは入れ替わる。対数変化率の年ごとの標準偏差で、ユーロ・ドルのほうが大きかった年を数えておく。


In [ ]:
usd_jpy_log = 100 * np.log(usd_jpy).diff()
eur_usd_log = 100 * np.log(eur_usd).diff()
sd_usd_jpy = usd_jpy_log.groupby(usd_jpy_log.index.year).std()
sd_eur_usd = eur_usd_log.groupby(eur_usd_log.index.year).std()
eur_larger = sd_eur_usd > sd_usd_jpy
print(f"ユーロ・ドルのほうが大きかった年：{eur_larger.sum()} 年")
print(list(eur_larger.index[eur_larger]))


26年のうち10年は、ユーロ・ドルのほうが大きく動いた。割合で比べると、2つの通貨の組のどちらが大きく動くかは年によって入れ替わる。


### 追加問題：2024年の月ごとの標準偏差

追加問題は、2024年の対数変化率について月ごとの標準偏差を求め、最も大きい月と最も小さい月を答える問いだった。2024年の対数変化率を `.groupby()` で月（`.index.month`）ごとにまとめ、`.std()` を付ければよい。


In [ ]:
g_2024 = log_pct.loc["2024"]
sd_month = g_2024.groupby(g_2024.index.month).std()
print(f"最も大きい月：{sd_month.idxmax()} 月（{sd_month.max():.3f}%、2024年全体の {sd_month.max() / g_2024.std():.2f} 倍）")
print(f"最も小さい月：{sd_month.idxmin()} 月（{sd_month.min():.3f}%）")


最も大きいのは8月の1.177%で、2024年全体の0.668%の1.76倍だった。8月には、2024年に1日の変化が最も大きかった8月5日（−5.72円）が入っている。最も小さい4月は0.268%で、同じ年の中でも月によって変動の大きさは4倍以上違った。

同じ計算を26年分で行えば、月ごとの変動の大きさが312か月並んだ時系列になる。この回では、この時系列を使って来月の変動の大きさを予想する。


## 月ごとの変動の大きさ

冒頭で振り返ったように、レートが大きく動いた日の次の日は、また大きく動きやすい。この性質を、月ごとにまとめた変動の大きさで調べよう。第6回の追加問題では、2024年の各月について、日々の対数変化率（前日からの変化を前日の値に対する割合で表した値）の標準偏差を求めた。同じ計算を2000〜2025年の26年分で行う。

年と月の両方で日付をまとめるには `.resample()` を使う。`log_pct.resample("ME")` は日付の列を月ごとにまとめ、その後に `.std()` を付けると各月の標準偏差を計算する。`"ME"` を指定すると、まとめた行には各月の最後の日付が付く。各月の標準偏差をその月の変動の大きさとし、`vol` という名前を付ける。


In [ ]:
vol = log_pct.resample("ME").std()
print(f"月数：{len(vol)} か月")
vol.loc["2024-06":"2024-10"]


2024年8月の行は1.177%で、第6回の追加問題で求めた8月の標準偏差と同じ値である。2000年1月から2025年12月までの312か月分を図に描こう。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(vol.index, vol, linewidth=0.8)
ax.set(title="月ごとの変動の大きさ（対数変化率の標準偏差、東京17時）", ylabel="%")
format_axis(ax)
fig.tight_layout()
plt.show()

print(f"最も大きい月：{vol.idxmax():%Y年%m月}（{vol.max():.3f}%）")
print(f"最も小さい月：{vol.idxmin():%Y年%m月}（{vol.min():.3f}%）")


変動の大きさは、最も大きい2008年10月の1.749%から最も小さい2019年4月の0.211%まで、8倍ほどの開きがある。図では2008〜2009年に大きい月が集まり、2019〜2021年には小さい月が多く並ぶ。変動の大きい月は大きい月のそばに、小さい月は小さい月のそばに現れやすい。

前月の値を横軸、今月の値を縦軸にとった散布図で、この並び方を確かめよう。`vol.shift(1)` は値を1つ下にずらして各行に前月の値を入れるので、前月の値と今月の値を同じ行に並べた表 `vols` が作れる。


In [ ]:
vols = pd.DataFrame({
    "前月の値": vol.shift(1),
    "今月の値": vol,
}).dropna()

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(vols["前月の値"], vols["今月の値"], "o", markersize=3)
ax.set(xlabel="前月の変動の大きさ（%）", ylabel="今月の変動の大きさ（%）",
       title="前月と今月の変動の大きさ（2000〜2025年）")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()

print(f"相関係数：{vols['前月の値'].corr(vols['今月の値']):.3f}")


点は右上がりに並び、相関係数は0.505である。冒頭で振り返った、日ごとの変化の大きさの相関係数0.166より大きく、前月の変動が大きいと今月の変動も大きくなりやすい。ただし点は大きく散らばっていて、前月の値から分かるのは今月の値のおおよその見当である。


## 前月の値で今月の値を説明する直線

散布図の点のそばを通る直線は

$$\text{今月の値} = a + b \times \text{前月の値}$$

と書け、$a$ を切片、$b$ を傾きという。傾き $b$ は、前月の値が高い月ほど今月の値がどれだけ高くなるかを表し、前月の値が0.1%高いと今月の値は平均して $b \times 0.1$% 高くなる。切片 $a$ は前月の値が0のときの値である。

点に最もよく合う直線は、各月の今月の値と直線から求めた値との差を2乗し、足し合わせた値が最も小さくなるように $a$ と $b$ を決めて求める。この決め方を最小二乗法という。差を2乗するのは、第6回で標準偏差を求めたときと同じく、マイナスの差もプラスに直して足し合わせるためである。このように、ある値を別の値の直線で説明する方法を回帰という。


## AR(1) モデル

前月の値で今月の値を説明する直線のように、ある系列の今の値をその系列の前の値で説明する回帰を自己回帰という。1つ前の値だけを使う自己回帰の式を AR(1) モデルと呼び、$t$ 期の値を $y_t$、1期前の値を $y_{t-1}$ として次のように書く。

$$y_t = c + \phi\, y_{t-1} + \varepsilon_t$$

$c$ は切片、$\phi$（ファイ）は傾きで、前の節の直線の $a$ と $b$ にあたる。$\varepsilon_t$（イプシロン）は $y_t$ のうち直線で説明できない部分で、$y_t$ と直線から求めた値との差を表す。AR は自己回帰の英語 autoregression の頭文字で、(1) は1期前の値まで使うことを表す。

AR(1) モデルで次の期を予想するときは、最新の値を $y_{t-1}$ に入れた $c + \phi\, y_{t-1}$ を予想にする。この予想は、傾き $\phi$ の値によって次のように変わる。

- $\phi = 1$、$c = 0$ なら、予想は前の値 $y_{t-1}$ と同じになる。冒頭で振り返った、今日のレートをそのまま次の日の予想にする方法がこれにあたる。
- $\phi = 0$ なら、予想は前の値によらず、いつも $c$ になる。前の値は予想の役に立たない。


## 傾き φ と系列の動き

$\phi$ の大きさで系列の動きがどう変わるかを、AR(1) モデルの式で作った系列で見ておこう。$c = 0$ とし、$\varepsilon_t$ には平均0、標準偏差1の乱数を使って、$\phi = 0.1$、$0.5$、$0.8$ の3つの系列を200期分作る。3つの系列には同じ $\varepsilon_t$ を使うので、系列の違いは $\phi$ の違いだけから生じる。

`np.random.default_rng(7)` は乱数を作る関数で、7のように数を決めておくと、何度実行しても同じ乱数になる。`np.zeros(200)` は0を200個並べた列で、`for` の繰り返しで $t = 1$ から順に $y_t = \phi\, y_{t-1} + \varepsilon_t$ を計算して入れていく。図のタイトルには、各系列の1期前の値との相関係数を `.autocorr()` で求めて表示する。


In [ ]:
rng = np.random.default_rng(7)
eps = rng.normal(size=200)
phis = [0.1, 0.5, 0.8]

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True, sharey=True)
for k in range(3):
    phi = phis[k]
    y = np.zeros(200)
    for t in range(1, 200):
        y[t] = phi * y[t - 1] + eps[t]
    r = pd.Series(y).autocorr()
    axes[k].plot(y, linewidth=0.8)
    axes[k].axhline(0, color="gray", linewidth=0.8)
    axes[k].set(title=f"φ = {phi}（1期前の値との相関係数 {r:.2f}）", ylabel="y")
    format_axis(axes[k], dates=False)
axes[2].set_xlabel("t")
fig.tight_layout()
plt.show()


φ = 0.1 の系列は0のまわりを細かく上下し、ある期の値が大きくても、次の期にはその影響がほとんど残らない。φ = 0.8 の系列では、t が20〜40 のあたりのように0から離れた値がしばらく続き、ゆっくり0へ戻る。大きな値の次は大きな値、小さな値の次は小さな値になりやすい。φ = 0.5 の系列はその中間である。1期前の値との相関係数は0.04、0.47、0.80で、φ が大きいほど大きい。

φ を0.95や −0.5 に替えて、描き直してみよう。


## AR(1) モデルの推定

月ごとの変動の大きさに AR(1) モデルを当てはめ、$c$ と $\phi$ を最小二乗法で求める。直線を求めるのには2000〜2019年の値だけを使い、2020〜2025年の72か月は予想を確かめるために残しておく。2020年以降の予想を確かめるのにその期間の値で直線を決めると、予想する時点では分からない値を使うことになるからである。

切片のある直線を求めるには、前月の値の列の左に、すべての行が1の列を加えておく。`sm.add_constant()` はこの列を加える関数で、加えた列には `const` という名前が付く。


In [ ]:
train_vol = vols.loc[:"2019"]
X = sm.add_constant(train_vol["前月の値"])
X.head(3)


`const` の列はすべて1になっている。式の切片 $c$ は $c \times 1$ と書けるので、この1の列に掛ける数として求まる。

`sm.OLS(今月の値, X)` は、今月の値を `X` の列で説明する直線を最小二乗法で求める準備をする関数で、`.fit()` を付けると実際に計算する。求めた切片と傾きは `.params` に入っていて、`["const"]` で切片、`["前月の値"]` で傾きを取り出せる。


In [ ]:
model_vol = sm.OLS(train_vol["今月の値"], X).fit()
c_vol = model_vol.params["const"]
phi_vol = model_vol.params["前月の値"]
print(f"切片 c：{c_vol:.3f}")
print(f"傾き φ：{phi_vol:.3f}")


求めた式は

$$\text{今月の値} = 0.287 + 0.507 \times \text{前月の値}$$

である。傾きは0.507で、前月の変動の大きさが0.1%高いと、今月の予想は0.05%ほど高くなる。「傾き φ と系列の動き」の図では、φ = 0.5 の系列にあたる。切片の0.287は前月の値が0のときの値で、この値そのものに意味はない。

求めた直線を、2000〜2019年の散布図に重ねて描こう。`ax.axline()` は、通る点と傾きを指定して直線を引く。灰色の線は前月の値と今月の値が等しい線で、前月の値をそのまま予想にする方法（$\phi = 1$、$c = 0$）にあたる。


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(train_vol["前月の値"], train_vol["今月の値"], "o", markersize=3)
ax.axline((0, c_vol), slope=phi_vol, color="C1", label="AR(1) モデルの直線")
ax.axline((0, 0), slope=1, color="gray", linewidth=0.8, label="前月の値と同じ")
ax.set(xlabel="前月の変動の大きさ（%）", ylabel="今月の変動の大きさ（%）",
       title="前月と今月の変動の大きさ（2000〜2019年）")
ax.legend(loc="upper left")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


オレンジ色の直線は灰色の線より傾きがゆるやかで、2本の線は前月の値が0.6%近くのところで交わる。前月の値がそれより高い月には、直線による予想は前月の値より低くなる。前月の値がそれより低い月には、予想は前月の値より高くなる。

2本の線が交わる位置を、2000〜2019年の平均と比べて確かめよう。前月の値をいくつか式に当てはめる。


In [ ]:
train_mean = vol.loc[:"2019"].mean()
print(f"2000〜2019年の平均：{train_mean:.3f}%")
for previous in [train_mean, 1.0, 0.3]:
    print(f"前月が {previous:.3f}% のときの予想：{c_vol + phi_vol * previous:.3f}%")


前月が平均と同じ0.585%なら、予想も0.584%とほぼ平均になる。前月が1.000%と平均より高ければ予想は0.795%で、前月の値と平均のほぼ中間にくる。前月が0.300%と低いときの予想0.440%も、前月の値と平均の中間あたりにある。傾きが0.5ほどの AR(1) モデルは、前月の値から平均の側へ半分ほど寄せた値を予想にする。

前月が平均と同じなら予想もほぼ平均になるのは、最小二乗法で求めた直線が、点の平均の位置を通るからである。


## 1か月先の変動の大きさの予想

2020〜2025年の各月について、3つの方法で予想を作る。AR(1) モデルの予想には、2019年までの値で求めた切片と傾きを使い続けて、前月の値を当てはめる。比べるのは、前月の値をそのまま予想にする方法と、2000〜2019年の平均を予想にする方法である。どの方法も、予想する月の値は使っていない。


In [ ]:
test_vol = vols.loc["2020":"2025"]
forecast_vol = pd.DataFrame({
    "今月の値": test_vol["今月の値"],
    "AR(1) モデル": c_vol + phi_vol * test_vol["前月の値"],
    "前月の値": test_vol["前月の値"],
    "学習期間の平均": train_mean,
})
forecast_vol.loc["2024-06":"2024-10"]


2024年7月は0.848%で、平均の0.585%より高かった。8月の予想は、前月の値による方法では0.848%、AR(1) モデルでは0.717%で、AR(1) モデルの予想は前月の値と平均の間にある。実際の8月は1.177%まで大きくなり、どちらの予想も届かなかった。9月の予想は前月の値で1.177%、AR(1) モデルで0.885%となり、実際の9月の0.851%には AR(1) モデルのほうが近かった。

2020〜2025年の72か月について、誤差の大きさの平均を3つの方法で比べよう。


In [ ]:
for name in ["AR(1) モデル", "前月の値", "学習期間の平均"]:
    error = (forecast_vol["今月の値"] - forecast_vol[name]).abs().mean()
    print(f"{name}：誤差の大きさの平均 {error:.3f}%")


AR(1) モデルの誤差の大きさの平均は0.161%で、前月の値の0.177%と学習期間の平均の0.195%のどちらより小さい。72か月を通すと、前月の値と平均の間をとる AR(1) モデルの予想が最もよく当たった。月ごとの予想と実際の値を図で比べよう。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(forecast_vol.index, forecast_vol["今月の値"], "o-", markersize=3, linewidth=0.8, label="実際の値")
ax.plot(forecast_vol.index, forecast_vol["AR(1) モデル"], linewidth=0.8, label="AR(1) モデルの予想")
ax.axhline(train_mean, color="gray", linewidth=0.8, label="学習期間の平均")
ax.set(title="月ごとの変動の大きさと1か月先の予想（2020〜2025年）", ylabel="%")
format_axis(ax)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()


オレンジ色の線の AR(1) モデルの予想は、青い線の実際の値を1か月遅れで追い、上下の動きは実際の値より小さい。2020年3月に1.2%を超えた次の4月は、実際の値が0.4%ほどに下がり、AR(1) モデルの予想は0.9%ほどだった。前月の値による予想なら1.2%を超えていたので、大きく動いた月の次の月には、平均の側へ寄せた AR(1) モデルの予想のほうが誤差が小さい。

2021年のように変動の小さい月が続いた時期には、AR(1) モデルの予想は平均の側へ寄せられて、実際の値より高めになり続けた。このような時期には、前月の値をそのまま使う予想のほうが実際の値に近い。


## 前日の変化に当てはめた AR(1) モデル

日々のレートの前日からの変化にも、同じ AR(1) モデルを当てはめる。`rate.diff()` は、各日のレートから前の観測日のレートを引いて、前日から何円動いたかを求める。各行に、その日の変化（当日の変化）と1つ前の観測日の変化（前日の変化）を並べた表 `changes` を作る。

直線を求めるのには2000〜2023年の値を使い、2024年は予想を確かめるために残す。2024年を残すのは、冒頭で振り返った、今日のレートをそのまま次の日の予想にする方法（2024年の誤差の大きさの平均0.70円）と比べるためである。`X` を作ってから `sm.OLS()` に渡した前の節の手順を、ここでは1行にまとめて書いている。


In [ ]:
change = rate.diff()
changes = pd.DataFrame({
    "前日の変化": change.shift(1),
    "当日の変化": change,
}).dropna()

train_change = changes.loc[:"2023"]
model_change = sm.OLS(train_change["当日の変化"], sm.add_constant(train_change["前日の変化"])).fit()
c_change = model_change.params["const"]
phi_change = model_change.params["前日の変化"]
print(f"切片 c：{c_change:.4f}")
print(f"傾き φ：{phi_change:.4f}")
print(f"前日に2円の円安だったときの当日の変化の予想：{c_change + phi_change * 2:.3f} 円／ドル")


傾きは −0.0142 で、ほぼ0である。冒頭で振り返った、ある日の変化と次の日の変化の相関係数 −0.019 と同じく、0に近い。前日に2円の円安だった次の日でも、当日の変化の予想は −0.022 円で、ほとんど動かないという予想になる。前日の変化が数円の大きさでも、予想は0円の近くにとどまる。「AR(1) モデル」の節の $\phi = 0$ の場合にあたり、「傾き φ と系列の動き」の図の φ = 0.1 の系列より、さらに前の値の影響が小さい。前日の変化は、次の日の変化の予想にほとんど役に立たない。

変化の予想を今日の値に足せば、次の日の値の予想になる。2024年について、今日の値による予想と誤差の大きさの平均を比べよう。`rate.shift(1)` で各日の前日の値（今日の値による予想）を取り出し、AR(1) モデルによる変化の予想を足す。


In [ ]:
test_change = changes.loc["2024"]
previous = rate.shift(1).loc["2024"]
actual = rate.loc["2024"]
pred_change = c_change + phi_change * test_change["前日の変化"]
error_naive = (actual - previous).abs().mean()
error_change = (actual - (previous + pred_change)).abs().mean()
print(f"今日の値による予想：{error_naive:.3f} 円／ドル")
print(f"前日の変化に当てはめた AR(1) モデル：{error_change:.3f} 円／ドル")


AR(1) モデルの予想の誤差の大きさの平均は0.694円で、今日の値による予想の0.696円とほぼ同じだった。冒頭の0.70円は、この0.696円を小数第2位までに丸めた値である。2つの予想の差は、1000ドルの両替でも数円にとどまる。日々のレートでは、前日の変化を使っても、今日の値による予想より誤差を小さくすることはほとんどできない。次の日のレートを動かすのは、雇用統計の発表のように前の日にはまだ分からなかった新しい情報で、前日の変化にはその情報が入っていないからである。


## レートの向きと変動の大きさ

月ごとの変動の大きさでは AR(1) モデルの傾きが0.507で、1か月先の予想は前月の値と平均のどちらより誤差が小さかった。日々のレートの変化では傾きが −0.0142 とほぼ0で、予想は今日の値による予想とほとんど変わらなかった。

レートが次の日に上がるか下がるかは、前の値から読めない。レートがどのくらい大きく動くかは、前の月の動きの大きさからある程度の見当がつく。変動の大きさの予想は、第11回と第12回でさらに調べる。


## 演習1：AR(1) モデルの傾きと予想

1. 問1の計算欄のコードは、FRBのニューヨーク正午のユーロ・ドル（ドル／ユーロ）について、本文と同じ手順で月ごとの変動の大きさを求め、2000〜2019年の値で AR(1) モデルを当てはめる。セルを実行し、表示された傾きを答えよ。また、その傾きをドル・円（東京17時）の0.507と比べ、ユーロ・ドルでも変動の大きい月の次の月は変動が大きくなりやすいといえるかを答えよ。
2. 次の(a)〜(c)は正しいか。正しいか誤りかを答え、それぞれ理由を1文で書け。
   - (a) 前日の変化に当てはめた AR(1) モデルの傾きは −0.0142 と負なので、前日に円安だった次の日は円高になると予想でき、今日の値による予想より誤差がずっと小さくなる。
   - (b) AR(1) モデルの傾き φ が0なら、予想は前の値によらず、いつも切片 c と同じ値になる。
   - (c) 月ごとの変動の大きさの AR(1) モデル（傾き0.507）では、前月の変動の大きさが学習期間の平均より高いとき、今月の予想は前月の値より高くなる。
3. AさんとBさんは、本文の AR(1) モデルの結果を自分の計画に使えるかを考えている。Aさんは明日1000ドルを両替する予定で、明日のレートが今日より上がるか下がるかを知りたい。Bさんは来月の旅行の予算を立てていて、来月の1か月の間に日々のレートがどのくらい大きく動きそうか（来月の変動の大きさ）を知りたい。本文の AR(1) モデルの結果は、2人のそれぞれの役に立つか。本文の傾きと誤差の大きさの平均を使って、3〜5文で説明せよ。

追加問題：本文で求めた月ごとの変動の大きさの AR(1) モデル（切片 `c_vol`、傾き `phi_vol`）と2025年12月の変動の大きさを使って、2026年1月の変動の大きさを予想せよ。さらに、その予想を前月の値として式に当てはめ、2026年2月の予想も求めよ。2つの予想は、2025年12月の値と2000〜2019年の平均（`train_mean`）のどちらに近づいていくかを答えよ。


In [ ]:
# 問1の計算欄：ユーロ・ドル（FRB）の月ごとの変動の大きさに AR(1) モデルを当てはめる
fed = load_fed_fx()
eur_log_pct = 100 * np.log(fed["eur_usd"].dropna()).diff()
eur_vol = eur_log_pct.resample("ME").std()
eur_vols = pd.DataFrame({
    "前月の値": eur_vol.shift(1),
    "今月の値": eur_vol,
}).dropna()
eur_train = eur_vols.loc[:"2019"]
model_eur = sm.OLS(eur_train["今月の値"], sm.add_constant(eur_train["前月の値"])).fit()
print(f"ユーロ・ドルの切片 c：{model_eur.params['const']:.3f}")
print(f"ユーロ・ドルの傾き φ：{model_eur.params['前月の値']:.3f}")


In [ ]:
# 追加問題の計算欄
# 2025年12月の変動の大きさは vol.loc["2025-12-31"] で取り出せる。


### 演習1の記入欄

- 問1：ユーロ・ドルの傾き、ドル・円の0.507との比較、変動の大きい月の次の月も変動が大きくなりやすいといえるか：
- 問2(a)：正しいか誤りか、その理由：
- 問2(b)：正しいか誤りか、その理由：
- 問2(c)：正しいか誤りか、その理由：
- 問3（3〜5文）：
- 追加問題：2026年1月と2月の予想、どちらに近づいていくか：


<details>
<summary>演習1のヒント</summary>

問1では、表示された傾きが0と1のどちらに近いか、ドル・円の0.507より大きいか小さいかを見よう。

問2の(a)は「前日の変化に当てはめた AR(1) モデル」の節で、前日に2円の円安だったときの予想と、誤差の大きさの平均を見よう。(b)は「AR(1) モデル」の節の予想の式に $\phi = 0$ を当てはめる。(c)は「AR(1) モデルの推定」の節で、前月が1.000%のときの予想を見よう。

問3では、Aさんの知りたいことには前日の変化に当てはめた AR(1) モデルが、Bさんの知りたいことには月ごとの変動の大きさの AR(1) モデルが対応する。それぞれの傾きと、今日の値（前月の値）による予想と比べた誤差の大きさの平均を使って説明しよう。

追加問題の予想は、本文で前月が1.000%のときの予想を求めたセルと同じ形の式で計算できる。2月の予想には、1月の予想を前月の値として当てはめる。

</details>


## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。

第6回 演習1の解説と演習1の問1で使う米国の為替レート（ドル・円、ユーロ・ドル）は [FRB H.10（為替）](https://www.federalreserve.gov/releases/h10/)による。
